# NOVAMART — Phase 6 Demand Forecasting

Target: daily `units_sold` at local SKU × location. Primary horizon: 14 days. The public source is **synthetic**. This notebook calls the reproducible pipeline; no inventory optimization occurs.

## 1. Problem, chronology, and data preparation
Random row splits would expose future patterns to training. Forecast origins are 14 days apart and each block is predicted from a frozen 28-day history. Same-day publisher-estimated demand and future realized fields are excluded.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd() if (Path.cwd()/"data/processed/daily_operations.csv").exists() else Path.cwd().parent
assert (ROOT/"data/processed/daily_operations.csv").exists()
sys.path.insert(0,str(ROOT/"src"))
from phase6_forecast import main, load_panel, make_rows, VAL_ORIGINS, TEST_ORIGINS, FEATURES
main()
manifest=json.loads((ROOT/"reports/phase6_manifest.json").read_text())
display(manifest)

## 2. Feature and leakage audit
Every lag and rolling statistic uses sales observed on or before its forecast origin. Future calendar is deterministic. SKU, location, and category are static. Price, discount, promotion, actual weather, inventory, publisher demand, and simulated deliveries are excluded because their future values or provenance cannot be justified. See `reports/PHASE6_LEAKAGE_AUDIT.md`.

In [ ]:
print("Included features:", FEATURES)
panel=load_panel()
block=make_rows(panel,[VAL_ORIGINS[0]])
assert (block.forecast_date>block.origin_date).all() and len(block)==1400
display(block[["origin_date","forecast_date","local_sku_id","last_sales","lag_7","mean_28","actual"]].head())

## 3. Baselines, models, and model selection
Compare last observation, seven-day mean, 28-day mean, Ridge, and HistGradientBoosting. The selected model uses validation WAPE with a one-point parsimony margin for Ridge. The final test is untouched until after selection. WAPE is total absolute error divided by total actual units; MAPE is avoided due to zero-sales days.

In [ ]:
comparison=pd.read_csv(ROOT/"reports/forecast_model_comparison.csv")
display(comparison.round(3))
display(Image(filename=str(ROOT/"images/forecasting/01_model_comparison.png")))

## 4. SKU and segment diagnostics
Out-of-sample errors combine validation and test blocks. WAPE can penalize a low-volume segment more than MAE; retain the worst series for honest planning.

In [ ]:
errors=pd.read_csv(ROOT/"reports/forecast_error_by_sku.csv")
display(errors.nsmallest(3,"WAPE_pct"))
display(errors.nlargest(3,"WAPE_pct"))
display(pd.read_csv(ROOT/"reports/forecast_by_category.csv").round(2))
display(pd.read_csv(ROOT/"reports/forecast_by_location.csv").round(2))
display(Image(filename=str(ROOT/"images/forecasting/02_actual_vs_forecast.png")))
display(Image(filename=str(ROOT/"images/forecasting/03_category_wape.png")))
display(Image(filename=str(ROOT/"images/forecasting/04_error_vs_variability.png")))

## 5. Forward 14-day forecast
Refit the selected Ridge model after evaluation, using eligible history through January 30, 2024. These fractional daily expectations become inputs for Phase 7, where policy and rounding must be specified.

In [ ]:
future=pd.read_csv(ROOT/"reports/demand_forecast_14d.csv")
summary=pd.read_csv(ROOT/"reports/demand_forecast_14d_summary.csv")
assert len(future)==1400 and future.local_sku_id.nunique()==100 and future.predicted_units.ge(0).all()
print("Dates:",future.forecast_date.min(),"through",future.forecast_date.max(),"Total:",future.predicted_units.sum())
display(summary.head())
display(Image(filename=str(ROOT/"images/forecasting/05_forward_example.png")))

## 6. Business interpretation and limits
Ridge test WAPE is 37.34%, only 0.35 percentage points lower than the 28-day mean. Its gain is modest. Stock-constrained sales, synthetic source generation, two annual cycles, and a five-origin test limit generalization. The SKU residual SD is historical out-of-sample error information, not a formal prediction interval. There are no replenishment recommendations or cost savings in this phase.